In [20]:
# ----------------------------- logging --------------------------
import logging
from sys import stdout
from datetime import datetime
import os

logging.basicConfig(
    level=logging.INFO,
    format=f"[%(asctime)s][%(levelname)s][{os.environ.get('USERNAME')}] %(message)s",
    stream=stdout,
    datefmt="%m-%d %H:%M:%S",
)
logging.info(datetime.now())

import numpy as np


# ####################################################################
def gauss_jordan(A: np.ndarray) -> np.ndarray:
    """Resuelve un sistema de ecuaciones lineales mediante el método de Gauss-Jordan.

    ## Parameters

    ``A``: matriz aumentada del sistema de ecuaciones lineales. Debe ser de tamaño n-by-(n+1), donde n es el número de incógnitas.

    ## Return

    ``solucion``: vector con la solución del sistema de ecuaciones lineales.

    """
    if not isinstance(A, np.ndarray):
        logging.debug("Convirtiendo A a numpy array.")
        A = np.array(
            A, dtype=float
        )  # convertir en float, porque si no, convierte en enteros
    assert A.shape[0] == A.shape[1] - 1, "La matriz A debe ser de tamaño n-by-(n+1)."
    n = A.shape[0]

    for i in range(0, n):  # loop por columna

        # --- encontrar pivote
        p = None  # default, first element
        for pi in range(i, n):
            if A[pi, i] == 0:
                # must be nonzero
                continue

            if p is None:
                # first nonzero element
                p = pi
                continue

            if abs(A[pi, i]) < abs(A[p, i]):
                p = pi

        if p is None:
            # no pivot found.
            logging.info(f"\n{A}")
            raise ValueError("No existe solución única.")

        if p != i:
            logging.info(f"Intercambiando filas {i} y {p}.")
            # swap rows
            _aux = A[i, :].copy()
            A[i, :] = A[p, :].copy()
            A[p, :] = _aux

        # --- Eliminación: loop por fila
        for j in range(n):
            if i == j:
                continue
            m = A[j, i] / A[i, i]
            A[j, i:] = A[j, i:] - m * A[i, i:]

        logging.info(f"\n{A}")

    if A[n - 1, n - 1] == 0:
        # Sin embargo, esto solo se accede al finalizar la matriz... Con todos los pivotes
        if A[n - 1, n] == 0:
            raise ValueError("Infinitas soluciones.")
        else:
            raise ValueError("Sin solución.")

    # --- Sustitución hacia atrás
    solucion = np.zeros(n)
    for i in range(n):
        solucion[i] = A[i, n] / A[i, i]

    return solucion

[07-21 17:38:22][INFO][None] 2026-07-21 17:38:22.865049


In [21]:
A = [
    [1, 2, 3, 4, 1],
    [2, 5, 6, 7, -2],
    [3, 6, 8, 9, 3],
    [4, 7, 9, 10, 4],
]

gauss_jordan(A)

[07-21 17:38:22][INFO][None] 
[[ 1.  2.  3.  4.  1.]
 [ 0.  1.  0. -1. -4.]
 [ 0.  0. -1. -3.  0.]
 [ 0. -1. -3. -6.  0.]]
[07-21 17:38:22][INFO][None] 
[[ 1.  0.  3.  6.  9.]
 [ 0.  1.  0. -1. -4.]
 [ 0.  0. -1. -3.  0.]
 [ 0.  0. -3. -7. -4.]]
[07-21 17:38:22][INFO][None] 
[[ 1.  0.  0. -3.  9.]
 [ 0.  1.  0. -1. -4.]
 [ 0.  0. -1. -3.  0.]
 [ 0.  0.  0.  2. -4.]]
[07-21 17:38:22][INFO][None] 
[[ 1.  0.  0.  0.  3.]
 [ 0.  1.  0.  0. -6.]
 [ 0.  0. -1.  0. -6.]
 [ 0.  0.  0.  2. -4.]]


array([ 3., -6.,  6., -2.])

# Resolver

In [ ]:
import numpy as np


# ####################################################################
def inv_matrix(A: np.ndarray) -> np.ndarray:
    """Inversión de una matriz cuadrada mediante método de Gauss-Jordan.
    ## Parameters
    ``A``: matriz cuadrada de tamaño n x n.

    ## Return
    ``A_inv``: matriz inversa de A.
    """
    if not isinstance(A, np.ndarray):
        logging.debug("Convirtiendo A a numpy array.")
        A = np.array(A, dtype=float)
    assert A.shape[0] == A.shape[1], "La matriz A debe ser cuadrada (n x n)."
    n = A.shape[0]

    I = np.eye(n)
    Aug = np.hstack([A, I])

    for i in range(n): 

        p = None 
        for pi in range(i, n):
            if Aug[pi, i] == 0:
                continue

            if p is None:
                p = pi
                continue

            if abs(Aug[pi, i]) < abs(Aug[p, i]):
                p = pi

        if p is None:
            logging.info(f"\n{Aug}")
            raise ValueError("La matriz no tiene inversa.")

        if p != i:
            logging.info(f"Intercambiando filas {i} y {p}.")
            _aux = Aug[i, :].copy()
            Aug[i, :] = Aug[p, :].copy()
            Aug[p, :] = _aux

        for j in range(n):
            if i == j:
                continue
            m = Aug[j, i] / Aug[i, i]
            Aug[j, i:] = Aug[j, i:] - m * Aug[i, i:]

        logging.info(f"\n{Aug}")

    for i in range(n):
        Aug[i, :] = Aug[i, :] / Aug[i, i]

    A_inv = Aug[:, n:]

    return A_inv

## Ejemplos
* Ejemplo 1

In [23]:
# La matriz A =
A = [
    [1, 2, 3, 4],
    [2, 5, 6, 7],
    [3, 6, 8, 9],
    [4, 7, 9, 10],
]
# tiene como inversa
# A_inv =[[ 0.5, -0.5, -1.5,  1.5],
#        [-0.5,  1.5, -1.5,  0.5],
#        [-1.5, -1.5,  3.5, -1.5],
#        [ 1.5,  0.5, -1.5,  0.5]]
inv_matrix(A)

[07-21 17:38:22][INFO][None] 
[[ 1.  2.  3.  4.  1.  0.  0.  0.]
 [ 0.  1.  0. -1. -2.  1.  0.  0.]
 [ 0.  0. -1. -3. -3.  0.  1.  0.]
 [ 0. -1. -3. -6. -4.  0.  0.  1.]]
[07-21 17:38:22][INFO][None] 
[[ 1.  0.  3.  6.  5. -2.  0.  0.]
 [ 0.  1.  0. -1. -2.  1.  0.  0.]
 [ 0.  0. -1. -3. -3.  0.  1.  0.]
 [ 0.  0. -3. -7. -6.  1.  0.  1.]]
[07-21 17:38:22][INFO][None] 
[[ 1.  0.  0. -3. -4. -2.  3.  0.]
 [ 0.  1.  0. -1. -2.  1.  0.  0.]
 [ 0.  0. -1. -3. -3.  0.  1.  0.]
 [ 0.  0.  0.  2.  3.  1. -3.  1.]]
[07-21 17:38:22][INFO][None] 
[[ 1.   0.   0.   0.   0.5 -0.5 -1.5  1.5]
 [ 0.   1.   0.   0.  -0.5  1.5 -1.5  0.5]
 [ 0.   0.  -1.   0.   1.5  1.5 -3.5  1.5]
 [ 0.   0.   0.   2.   3.   1.  -3.   1. ]]


array([[ 0.5, -0.5, -1.5,  1.5],
       [-0.5,  1.5, -1.5,  0.5],
       [-1.5, -1.5,  3.5, -1.5],
       [ 1.5,  0.5, -1.5,  0.5]])

* Ejemplo 2

In [24]:
# La matriz A =
A = [
    [4, 4, 5, 1],
    [3, 4, 2, 2],
    [2, 1, 4, 1],
    [3, 2, 5, 4],
]
# tiene como inversa
# A_inv =[[-34.,  31.,  52., -20.],
#         [ 19., -17., -29.,  11.],
#         [ 12., -11., -18.,   7.],
#         [  1.,  -1.,  -2.,   1.]]
inv_matrix(A)

[07-21 17:38:22][INFO][None] Intercambiando filas 0 y 2.
[07-21 17:38:22][INFO][None] 
[[ 2.   1.   4.   1.   0.   0.   1.   0. ]
 [ 0.   2.5 -4.   0.5  0.   1.  -1.5  0. ]
 [ 0.   2.  -3.  -1.   1.   0.  -2.   0. ]
 [ 0.   0.5 -1.   2.5  0.   0.  -1.5  1. ]]
[07-21 17:38:22][INFO][None] Intercambiando filas 1 y 3.
[07-21 17:38:22][INFO][None] 
[[  2.    0.    6.   -4.    0.    0.    4.   -2. ]
 [  0.    0.5  -1.    2.5   0.    0.   -1.5   1. ]
 [  0.    0.    1.  -11.    1.    0.    4.   -4. ]
 [  0.    0.    1.  -12.    0.    1.    6.   -5. ]]
[07-21 17:38:22][INFO][None] 
[[  2.    0.    0.   62.   -6.    0.  -20.   22. ]
 [  0.    0.5   0.   -8.5   1.    0.    2.5  -3. ]
 [  0.    0.    1.  -11.    1.    0.    4.   -4. ]
 [  0.    0.    0.   -1.   -1.    1.    2.   -1. ]]
[07-21 17:38:22][INFO][None] 
[[  2.    0.    0.    0.  -68.   62.  104.  -40. ]
 [  0.    0.5   0.    0.    9.5  -8.5 -14.5   5.5]
 [  0.    0.    1.    0.   12.  -11.  -18.    7. ]
 [  0.    0.    0.   -1.   -1.

array([[-34.,  31.,  52., -20.],
       [ 19., -17., -29.,  11.],
       [ 12., -11., -18.,   7.],
       [  1.,  -1.,  -2.,   1.]])

## Ejercicios

* Ejercicio 1

In [25]:
A = [[2, -3], [-1, 1]]
inv_matrix(A)

[07-21 17:38:22][INFO][None] Intercambiando filas 0 y 1.
[07-21 17:38:22][INFO][None] 
[[-1.  1.  0.  1.]
 [ 0. -1.  1.  2.]]
[07-21 17:38:22][INFO][None] 
[[-1.  0.  1.  3.]
 [ 0. -1.  1.  2.]]


array([[-1., -3.],
       [-1., -2.]])

* Ejercicio 2

In [26]:
A = [
    [4, 0, 0, 5],
    [1, 0, 4, 0],
    [3, 4, 1, 3],
    [1, 3, 3, 0],
]
inv_matrix(A)

[07-21 17:38:22][INFO][None] Intercambiando filas 0 y 1.
[07-21 17:38:22][INFO][None] 
[[  1.   0.   4.   0.   0.   1.   0.   0.]
 [  0.   0. -16.   5.   1.  -4.   0.   0.]
 [  0.   4. -11.   3.   0.  -3.   1.   0.]
 [  0.   3.  -1.   0.   0.  -1.   0.   1.]]
[07-21 17:38:22][INFO][None] Intercambiando filas 1 y 3.
[07-21 17:38:22][INFO][None] 
[[  1.           0.           4.           0.           0.
    1.           0.           0.        ]
 [  0.           3.          -1.           0.           0.
   -1.           0.           1.        ]
 [  0.           0.          -9.66666667   3.           0.
   -1.66666667   1.          -1.33333333]
 [  0.           0.         -16.           5.           1.
   -4.           0.           0.        ]]
[07-21 17:38:22][INFO][None] 
[[ 1.          0.          0.          1.24137931  0.          0.31034483
   0.4137931  -0.55172414]
 [ 0.          3.          0.         -0.31034483  0.         -0.82758621
  -0.10344828  1.13793103]
 [ 0.          0

array([[-36.,  45.,  60., -80.],
       [  3.,  -4.,  -5.,   7.],
       [  9., -11., -15.,  20.],
       [ 29., -36., -48.,  64.]])

* Ejercicio 3

In [27]:
A = [
    [0, 0, 0, 0, 0, 0, 1, -1],
    [0, 1, -1, 1, 0, -1, 0, 1],
    [-1, -1, 0, 0, 2, 1, 0, 0],
    [-1, -1, -1, 1, 2, 0, 0, 1],
    [-1, 1, 1, 0, -1, -1, 0, 2],
    [0, 1, 0, 0, -1, -1, 0, 0],
    [1, -1, -1, 1, 2, 1, 0, 2],
    [2, 0, 0, 0, 0, 1, 2, 0],
]
inv_matrix(A)

[07-21 17:38:22][INFO][None] Intercambiando filas 0 y 2.
[07-21 17:38:22][INFO][None] 
[[-1. -1.  0.  0.  2.  1.  0.  0.  0.  0.  1.  0.  0.  0.  0.  0.]
 [ 0.  1. -1.  1.  0. -1.  0.  1.  0.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  1. -1.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  1.  0. -1.  0.  1.  0.  0. -1.  1.  0.  0.  0.  0.]
 [ 0.  2.  1.  0. -3. -2.  0.  2.  0.  0. -1.  0.  1.  0.  0.  0.]
 [ 0.  1.  0.  0. -1. -1.  0.  0.  0.  0.  0.  0.  0.  1.  0.  0.]
 [ 0. -2. -1.  1.  4.  2.  0.  2.  0.  0.  1.  0.  0.  0.  1.  0.]
 [ 0. -2.  0.  0.  4.  3.  2.  0.  0.  0.  2.  0.  0.  0.  0.  1.]]
[07-21 17:38:22][INFO][None] 
[[-1.  0. -1.  1.  2.  0.  0.  1.  0.  1.  1.  0.  0.  0.  0.  0.]
 [ 0.  1. -1.  1.  0. -1.  0.  1.  0.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  1. -1.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  1.  0. -1.  0.  1.  0.  0. -1.  1.  0.  0.  0.  0.]
 [ 0.  0.  3. -2. -3.  0.  0.  0.  0. -2. -1.  0.  1.  0.  0.  0.]
 [ 0.  0.  

array([[ 2., -1., -0., -1., -0.,  2.,  2., -1.],
       [ 0.,  1.,  1., -1.,  0.,  0.,  0.,  0.],
       [ 6., -1., -0., -3.,  1.,  1.,  4., -3.],
       [ 6.,  1., -1., -3.,  1., -3.,  3., -3.],
       [ 2., -1.,  1., -1., -0.,  3.,  2., -1.],
       [-2.,  2., -0., -0., -0., -4., -2.,  1.],
       [-1.,  0.,  0.,  1.,  0.,  0., -1.,  1.],
       [-2.,  0.,  0.,  1.,  0.,  0., -1.,  1.]])

* Ejercicio 4

In [28]:
A = [
    [1, 0, 0, 0, -1, 0, 0, -1, 1, -1],
    [1, 1, 0, -1, -1, 1, 0, 0, 1, -1],
    [-1, 0, -1, 0, 0, 0, -1, 1, 0, 0],
    [0, 0, -1, 0, -1, -1, 1, 0, 1, 0],
    [-1, 0, 0, -1, 1, 1, 1, 1, 0, -1],
    [1, 0, 0, 1, -1, -1, -1, 1, -1, 0],
    [1, 1, 1, 0, 1, 0, -1, -1, -1, 1],
    [1, 1, 1, 1, 0, 0, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 0, -1, -1, 0, 0],
    [0, 0, -1, -1, -1, 0, 1, 1, 1, -1],
]
inv_matrix(A)

[07-21 17:38:22][INFO][None] 
[[ 1.  0.  0.  0. -1.  0.  0. -1.  1. -1.  1.  0.  0.  0.  0.  0.  0.  0.
   0.  0.]
 [ 0.  1.  0. -1.  0.  1.  0.  1.  0.  0. -1.  1.  0.  0.  0.  0.  0.  0.
   0.  0.]
 [ 0.  0. -1.  0. -1.  0. -1.  0.  1. -1.  1.  0.  1.  0.  0.  0.  0.  0.
   0.  0.]
 [ 0.  0. -1.  0. -1. -1.  1.  0.  1.  0.  0.  0.  0.  1.  0.  0.  0.  0.
   0.  0.]
 [ 0.  0.  0. -1.  0.  1.  1.  0.  1. -2.  1.  0.  0.  0.  1.  0.  0.  0.
   0.  0.]
 [ 0.  0.  0.  1.  0. -1. -1.  2. -2.  1. -1.  0.  0.  0.  0.  1.  0.  0.
   0.  0.]
 [ 0.  1.  1.  0.  2.  0. -1.  0. -2.  2. -1.  0.  0.  0.  0.  0.  1.  0.
   0.  0.]
 [ 0.  1.  1.  1.  1.  0.  1.  2. -1.  1. -1.  0.  0.  0.  0.  0.  0.  1.
   0.  0.]
 [ 0.  1.  1.  1.  2.  0. -1.  0. -1.  1. -1.  0.  0.  0.  0.  0.  0.  0.
   1.  0.]
 [ 0.  0. -1. -1. -1.  0.  1.  1.  1. -1.  0.  0.  0.  0.  0.  0.  0.  0.
   0.  1.]]
[07-21 17:38:22][INFO][None] 
[[ 1.  0.  0.  0. -1.  0.  0. -1.  1. -1.  1.  0.  0.  0.  0.  0.  0.  0.
   0.  0.]
 [ 0

array([[ 14.,  -8.,   9.,  -4.,   0.,  -4.,   9.,   7.,  -8.,   3.],
       [ -2.,   2.,  -1.,   2.,   1.,   1.,  -1.,  -1.,   1.,  -2.],
       [-27.,  14., -18.,   5.,  -2.,   7., -17., -13.,  16.,  -2.],
       [ 12.,  -6.,   8.,  -2.,   1.,  -3.,   7.,   6.,  -7.,  -0.],
       [  6.,  -4.,   4.,  -2.,   0.,  -2.,   4.,   3.,  -3.,   2.],
       [ 18.,  -9.,  12.,  -4.,   1.,  -5.,  11.,   9., -11.,   1.],
       [  8.,  -4.,   5.,  -1.,   1.,  -2.,   5.,   4.,  -5.,  -0.],
       [ -5.,   2.,  -3.,   0.,  -1.,   1.,  -3.,  -2.,   3.,   1.],
       [-11.,   5.,  -7.,   1.,  -2.,   2.,  -7.,  -5.,   7.,   1.],
       [  1.,  -1.,   1.,  -1.,  -1.,  -1.,   1.,   1.,  -1.,   1.]])